# Day 25 — Tool authorization (PolicyGate)

The most important agent-security idea: **deny by default**. An agent identity holds a
small set of **scopes**; a tool runs only if its required scope is granted. High-blast
tools need **human approval**, and every decision is **audited**.

This is the heart of the Phase 9 capstone. Uses
[`shared.policy.PolicyGate`](../../shared/policy.py). ✅ Runs offline.

In [ ]:
# ▶ Run this first — puts the repo root on sys.path so `shared/` imports work.
import sys, pathlib
root = pathlib.Path.cwd()
while root != root.parent and not (root / "shared" / "llm.py").exists():
    root = root.parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
print("repo root on sys.path:", root)

## 🔬 Your turn

Fill in the `TODO`s, then run. The solution is below — try first.

In [ ]:
from shared.policy import PolicyGate, PolicyDenied
from shared.tools import calculator

TOOL_SCOPES = {"calculator": "math:use", "delete_files": "fs:admin"}
gate = PolicyGate(tool_scopes=TOOL_SCOPES, granted={"math:use"})  # NOT fs:admin

def run_tool(name, arg):
    # TODO: gate.authorize(name, arg) inside try/except PolicyDenied -> return f"DENIED: {exc}";
    #       on success: return calculator(arg) if name == "calculator" else "(did nothing)"
    raise NotImplementedError

# print(run_tool("calculator", "2+2"))
# print(run_tool("delete_files", "/"))
# gate.audit.dump()

## 🔒 Solution

One correct way to do it.

In [ ]:
from shared.policy import PolicyGate, PolicyDenied
from shared.tools import calculator

TOOL_SCOPES = {"calculator": "math:use", "delete_files": "fs:admin"}
gate = PolicyGate(tool_scopes=TOOL_SCOPES, granted={"math:use"})

def run_tool(name, arg):
    try:
        gate.authorize(name, arg)
    except PolicyDenied as exc:
        return f"DENIED: {exc}"
    return calculator(arg) if name == "calculator" else "(did nothing)"

print(run_tool("calculator", "2+2"))     # allowed
print(run_tool("delete_files", "/"))     # denied: missing scope
print(run_tool("exfiltrate", "secrets")) # denied: unknown tool
print("\n--- audit trail ---")
gate.audit.dump()